# SHIPP Stage 13 — Marketplace Analytics

**Owner:** Raneem  
**Priority:** P0  
**Current gate:** 13.0 — Analytics CDF source readiness

## Goal

Verify that the two operational Agent-write tables:

- `shipp.saved_items`
- `shipp.agent_activity`

are arriving in Bronze through Lakebase CDF before we build
`gold_marketplace_metrics`.

**This gate is read-only.** It does not create or modify analytics data.

### PASS condition

Both expected Bronze CDF tables must:

1. exist,
2. contain at least one row,
3. expose the SHIPP Lakebase CDF ordering metadata:
   - `_pg_change_type`
   - `_pg_lsn`
   - `_sort_by`


In [0]:
# MAGIC %run ../common/shipp_silver_lib

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.0
# Verify Analytics CDF sources.
# READ ONLY — this cell does not create or modify data.
# -------------------------------------------------------------------

BRONZE_SAVED_ITEMS_HISTORY = (
    f"{CATALOG}.{BRONZE_SCHEMA}.lb_saved_items_history"
)

BRONZE_AGENT_ACTIVITY_HISTORY = (
    f"{CATALOG}.{BRONZE_SCHEMA}.lb_agent_activity_history"
)

ANALYTICS_CDF_SOURCES = {
    "saved_items": BRONZE_SAVED_ITEMS_HISTORY,
    "agent_activity": BRONZE_AGENT_ACTIVITY_HISTORY,
}

REQUIRED_CDF_COLUMNS = {
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
}

print("=== STAGE 13 — ANALYTICS SOURCE READINESS ===")

source_results = {}

for source_name, table_name in ANALYTICS_CDF_SOURCES.items():
    print()
    print("Source:", source_name)
    print("Table :", table_name)

    exists = table_exists(table_name)
    print("Exists:", exists)

    source_results[source_name] = {
        "table": table_name,
        "exists": exists,
        "rows": None,
        "missing_cdf_columns": None,
    }

    if not exists:
        print("BLOCKED — expected Lakebase CDF history table does not exist.")
        continue

    df = spark.table(table_name)
    row_count = df.count()

    print("Rows  :", row_count)
    print("Columns:", df.columns)
    df.printSchema()

    missing_cdf_columns = sorted(
        REQUIRED_CDF_COLUMNS - set(df.columns)
    )

    source_results[source_name]["rows"] = row_count
    source_results[source_name]["missing_cdf_columns"] = (
        missing_cdf_columns
    )

    assert not missing_cdf_columns, (
        f"BLOCKED: {table_name} is missing required "
        f"CDF metadata columns: {missing_cdf_columns}"
    )

    assert row_count > 0, (
        f"BLOCKED: {table_name} exists but contains 0 rows. "
        "Stage 13 needs real Agent-write change evidence before "
        "building Gold analytics."
    )

    change_types = [
        row["_pg_change_type"]
        for row in (
            df.select("_pg_change_type")
              .distinct()
              .orderBy("_pg_change_type")
              .collect()
        )
    ]

    print("Change types:", change_types)
    print("Latest CDF rows:")

    display(
        df.orderBy(
            F.col("_pg_lsn").desc(),
            F.col("_sort_by").desc(),
        ).limit(10)
    )


missing_sources = [
    details["table"]
    for details in source_results.values()
    if not details["exists"]
]

assert not missing_sources, (
    "STAGE 13 BLOCKED — missing CDF source(s): "
    + ", ".join(missing_sources)
)

print()
print(
    "PASS Gate 13.0 — saved_items and agent_activity "
    "CDF histories exist, contain real rows, and expose "
    "the required ordering metadata."
)

In [0]:
# -------------------------------------------------------------------
# Gate 13.0 summary
# -------------------------------------------------------------------

gate_13_0_summary = {
    "stage": "13_analytics",
    "gate": "13.0_cdf_source_readiness",
    "saved_items_history": BRONZE_SAVED_ITEMS_HISTORY,
    "agent_activity_history": BRONZE_AGENT_ACTIVITY_HISTORY,
    "saved_items_rows": source_results["saved_items"]["rows"],
    "agent_activity_rows": source_results["agent_activity"]["rows"],
    "status": "PASS",
}

print(json.dumps(gate_13_0_summary, indent=2, default=str))